## <center><u><span style="color:blue">Job Validator</span></u></center>
### <u><span style="color:coral">Background</span></u>:
#### It's become clear to me that not only are job applicants facing an uphill battle in setting themselves up for success with every job application, but they are also faced with an uneasy feeling that the job they're adapting their resume for... may not be legitimate.
#### According to <span style="color:blue">LinkedIn</span> itself, up to (<i>get this</i>) <span style="color:red">[6 out of every 10</span> jobs posted are fake](https://www.linkedin.com/pulse/reality-fake-job-listings-linkedin-what-you-need-know-arani-ramnc/).
#### That's just flippin' ridiculous.
#### Listing things like <span style="color:deeppink">Lead Generation</span>, <span style="color:deeppink">Company Image Boost</span>, and - of course - <span style="color:deeppink">Data Harvesting</span> as reasons to why companies do this type of stuff, it just becomes another layer of nausea the job seeker has to deal with in what is already a humiliating, arduous process.
#### So I figured, "What good is a a resume optimizer without a tool that checks for the legitimacy of the job postings?"
#### And here we are.
***
### <span style="color:blue">Step 1: Imports</span>
#### <u>These are the tools we'll need for this program to run. Loads of them, so here is a breakdown of what they do:
##### <b><span style="color:green">requests</span></b> - 
##### <b><span style="color:green">bs4</span></b> / <b><span style="color:green">BeautifulSoup</span></b> -
##### <b><span style="color:green">re</span></b> -
##### <b><span style="color:green">time</span></b> -
##### <b><span style="color:green">urllib.parse</span></b> / <b><span style="color:green">urlparse</span></b> -
##### <b><span style="color:green">typing</span></b> / <b><span style="color:green">Optional</span></b> -
##### <b><span style="color:green">fuzzywuzzy</span></b> / <b><span style="color:green">fuzz, process</span></b> -
##### <b><span style="color:green">selenium</span></b> / <b><span style="color:green">webdriver</span></b> -
##### <b><span style="color:green">selenium.webdriver.chrome.options</span></b> / <b><span style="color:green">Options</span></b> -


In [1]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse
from typing import Optional
from fuzzywuzzy import fuzz, process
# for Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# for Chrome
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options as ChromeOptions
from webdriver_manager.chrome import ChromeDriverManager
# for Firefox
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from webdriver_manager.firefox import GeckoDriverManager
# for Edge
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from webdriver_manager.microsoft import EdgeChromiumDriverManager

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


***
### <span style="color:blue">Step 2: Functions - A Broad Overview</span>
##### <b><u><span style="color:blue ; font-family:menlo">get_webdriver</span></u></b>
> ##### A function that sets up the bulk of the code to be used by either <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b>, <b><span style="color:cornflowerblue">Saf</span><span style="color:red">a</span><span style="color:cornflowerblue">ri</span></b>, <b><span style="color:blue">Ed</span><span style="color:chartreuse">ge</span></b>, or <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b>. 
##### <b><u><span style="color:blue ; font-family:menlo">is_valid_url</span></u></b>
> ##### A structure check to see if a given piece of text looks like a legitimate URL / web address. It's set up to receive that familiar string of characters indicative of a url (eg: 'https://www.linkedin.com'). If that string of characters looks good, great! If not, <b><span style="color:green ; font-family:menlo">False</span>.
##### <b><u><span style="color:blue ; font-family:menlo">find_careers_page</span></u></b> 
> ##### Kind of our bread-and-butter function in this process. It's a pretty complicated function (for me it was, anyways), and - full disclosure - I got some help from <span style="color:blue">Gemini</span> on redirects and how to handle them.
***


### <span style="color:blue">Step 2a: the 'get_webdriver' function</span>
> ##### because not everyone uses Chrome, this function sets up the user's chosen browser ("web_driver") to work with Selenium Webdriver, which is (<i>very generally speaking</i>) a tool that acts like a remote control for your tv: <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b> remotes work with <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b> TVs, <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b> remotes work with <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b> TVs, etc.

In [2]:
def get_webdriver(browser_name: str): 
    """
    This function sets up the 'browser_name' in which the code will be running. 
    We're aiming at the four main browsers people use: Chrome, Safari, Edge, and Firefox
    """
    # because no browser is set up yet:
    driver = None
    if browser_name.lower() == "chrome":
        options = ChromeOptions()
        # run chrome in the background without showing user - prevent a headache
        options.add_argument("--headless")
        # since we're running the app on Render and we need Chrome to start porperly
        options.add_argument("--no-sandbox")
        # reduce crash-rate
        options.add_argument("--disable-dev-shm-usage")
        # download the chrome webdriver with the ChromeDriverManager - see imports
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    elif browser_name.lower() == "firefox":
        options = FirefoxOptions()
        options.add_argument("--headless")
        # download the firefox webdriver with GeckoDriverManager - see imports
        driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()), options=options)
    elif browser_name.lower() == "edge":
        options = EdgeOptions()
        # edge runs in the background the old-fashioned way where you have to declare 'headless'
        options.add_argument("--headless")
        # download the edge webdriver with EdgeChromiumDriverManager - see imports
        driver = webdriver.Edge(service=EdgeService(EdgeChromiumDriverManager().install()), options=options)
    # Safari's special: if you're running it, you have no other choice but to open it as a new window
    elif browser_name.lower() == "safari":
        driver = webdriver.Safari()
        print("Using Safari means a visible Safari browser will launch on your Mac.")
    else:
        raise ValueError(f"Sorry! {browser_name} is not supported. Try 'Chrome,' 'Firefox', 'Edge', or 'Safari' instead.")
    return driver

#### <span style="color:blue">Step 2b: the 'is_valid_url' function</span>
> ##### answers the question: am I getting a real url with my request?

In [3]:
def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

#### <span style="color:blue">Step 2c: the 'find_careers_page' function - doozy #1</span>
> ##### <b><u>Details regarding this function</u>:</b>
> ##### 1.) it lowercases company names, removes spaces, and strips special characters from the company's name;
> ##### 2.) it looks for websites that may not have that '.com' at the end (like a '.net,' or '.org');
> ##### 3.) makes the app look like an actual web browser for the server;
> ##### 4.) checks to make sure the URL is valid and what to do if it is not;
> ##### 5.) handles redirects, both server and client-side (thanks, <span style="color:blue">Gemini</span>!);
> ##### 6.) employs <b><span style="color:cornflowerblue">Beautiful</span><span style="color:goldenrod">Soup</span></b> to scrape the HTML / XML for any signs that the server is running any redirect code on my machine (client-side) - HTML or JavaScript; and
> ##### 7.) it creates a URL path and checks it for careers

In [4]:
def find_careers_page(company_name: str) -> Optional[str]:
    
    # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
    slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())

    # not everything's a '.com'
    possible_base_urls = [f"https://www.{slugged_company}.com", 
                          f"https://www.{slugged_company}.net", 
                          f"https://www.{slugged_company}.org"]
    
    # typical career posting paths & subdomains
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions", "/job-search"]
    possible_subdomains = ["careers.", "jobs."]

    # make the server believe we're one of the more popular clients / browsers
    # -> also, helps avoid blocks / captchas
    headers = {
        'User-Agent': (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
            )
        }
    
    # loop for legitimacy and domain building
    
    for base_url_attempt in possible_base_urls:
        # stop the loop if the 'possible_base_url' seems wonky and skip to the try block
        if not is_valid_url(base_url_attempt):
            continue
        # our plan if the 'possible_base_url' doesn't seem right
        try:
            # Send request to the main company URL and give it 7 seconds to respond
            response = requests.get(base_url_attempt, headers=headers, timeout=7)
            
            # No response after 7 seconds, check to see if we got an error message (like a 404 or 500 error)
            response.raise_for_status()

            # did the url redirect / somehow change after our request
            we_got_redirected = response.url.lower() != base_url_attempt.lower()
            
            # does that redirect have the career keywords we want?
            career_keyword_in_redirect = any (
                keyword in response.url.lower()
                for keyword in ["careers", "jobs", "opportunities"]
            )

            # if that redirect has any of our 'career' keywords, then it's likely the page we want and we can stop searching
            if we_got_redirected and career_keyword_in_redirect:
                return response.url

            # check the HTML to see if server is running JavaScript redirects in my browser 
            # -> important b/c 'requests' can't do JavaScript
            soup = BeautifulSoup(response.text, 'html.parser')

            # get soup to find any <meta> refresh HTML tags (look like '<meta http-equiv="refresh">')
            # -> we're looking for are 'http-equiv' that matches a case-insensitive pattern of 'refresh'
            meta_refresh = soup.find('meta', attrs={'http-equiv': re.compile(r'refresh', re.I)})
            
            # if soup finds a <meta> refresh tag with a 'content' attribute:
            if meta_refresh and 'content' in meta_refresh.attrs:
                # take that content text and store it as 'content'
                content = meta_refresh['content']
                # find anything in 'content' that looks like 'url=whatever' and store it 
                match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
                # if we get a match, store it as 'redirect_url'
                if match:
                    redirect_url = match.group(2)
                    
                    # break the redirect down into its components; and 
                    # change the relative URL path into to an absolute URL path
                    # -> '.netloc' == 'network location" == domain
                    if not urlparse(redirect_url).netloc:
                        redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                    
                    # If the redirected URL looks valid; AND
                    # the redirect contains "careers" or "jobs", give us the redirect url
                    if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                        return redirect_url

            # JavaScript tag check -> Soup search for all occurences of '<script>' tags in the HTML 
            script_tags = soup.find_all('script')

            # loop through the tags we found
            for script in script_tags:
                # check for and read the string content between the '<script></script>' tags
                # using regex to look to match the'window.location.href' string that indicates redirects
                if script.string:
                    js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    
                    # if there is a match, go to it (the redirect URL)
                    if js_match:
                        redirect_url = js_match.group(1)
                        # change relative URLs to absolute URLs - like the '.netloc' note above
                        if not urlparse(redirect_url).netloc:
                            redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                        
                        # If the redirected URL is valid and contains "careers" or "jobs", return it
                        if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                            return redirect_url

       # what to do about that bad url from the beginning (see 'try')
        except requests.exceptions.RequestException:
            
            # If the initial request to the base URL fails, try the very next base domain or path w/ a 1-second delay
            time.sleep(1)
            continue
    # create a url with the main url and 'extend' the possible paths:
    # -> turn 'https://www.bobsbeds.com' into 'https://www.bobsbeds.com/careers' 
    potential_urls = [base_url_attempt + path for path in possible_paths]
    
    # kind of like the above when adding paths to the base URL,
    # but this time with subdomains
    # -> building the 'potential_urls' list with '.extend' instead of '.append' so we can add ALL the URLs (not just one)
    potential_urls.extend([
        # replace 'www' with the subdomain - end up with something like 'https://careers.google.com' instead of 'https://www.google.com'
        base_url_attempt.replace("www.", subdomain)
        for subdomain in possible_subdomains
        # 
        if is_valid_url(base_url_attempt.replace("www.", subdomain) + possible_paths[0])
    ])
    potential_urls.append(f"https://{slugged_company}.com/careers")

    # for loop to check possible career pages
    for url in potential_urls:
        # "if" the address is properly formatted
        if is_valid_url(url):
            try:
                response = requests.get(url, headers=headers, timeout=5)
                response.raise_for_status()
                if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
                    return response.url
            # if it is not...
            except requests.exceptions.RequestException:
                time.sleep(1)
                
    # if we can't find or architect an actual career page:
    return None

### <span style="color:blue">Step 2d: the 'sanitize_title' function</span>
> ##### takes the job title character string and uses Regex to remove all the non-alphanumeric characters and whitespaces

In [5]:
def sanitize_title(title: str) -> str:
    # remove anything not a-z, A-Z, the digits 0-9, whitespaces, or pipes
    title = re.sub(r'[^a-zA-Z0-9\s\-\|]', '', title)
    # remove extra whitespace (eg: take "Bob's  Beds" and make it "bobsbeds" 
    title = re.sub(r'\s+', ' ', title).strip()
    # lowercase for URL
    return title.lower()

### <span style="color:blue">Step 2e: the 'validate_job_posting' function - doozy #2</span>
##### <b>This function takes the specific job title and company name, checks for that job posting in your preferred browser (defaults to Chrome), and gives us a response by:</b>
> ##### 1.) Using that extensive <span style="color:blue ; font-family:menlo">'find_careers_page'</span> function to find the most likely career page URL and letting us know if it was not found;
> ##### 2.) Incorporating the <span style="color:blue ; font-family:menlo">'get_webdriver'</span> function to allow you to use your preferred browser;
> ##### 3.) Slugs the career title we're looking ('Assistant Manager' becomes 'assistantmanager') for and scapes the URL we found for it; and
> ##### 4.) Creates a list of potential job titles found inside the common HTML tags and compares them to the job we're looking for using fuzzy matching and scoring job title similarities
 

In [6]:
def validate_job_posting(job_title: str, company_name: str, browser_name: str = "chrome") -> str:
    """
    Checks if the given job_title exists / can be closely matched with any listings on the company's job website,
    and returns a percentage confidence rating.
    """
    careers_url = find_careers_page(company_name)
    if not careers_url:
        # 'return' to stop the app since we couldn't find any career page
        return "Sorry. We couldn't automatically find the company's careers page. You may have to check manually."

    # try block to run the 'get_webdriver' function to initialize the driver that was returned
    # in this function, it defaults to Chrome
    try:
        driver = get_webdriver(browser_name)
    # in case we get that ValueError from 'get_webdriver:'
    except ValueError as e:
        return f"❌ Browser setup error: {e}"
    # if there are any other problems - like the manager can't download the driver:
    except Exception as e:
        return f"❌ Could not initialize WebDriver: {e}. Make sure the correct WebDriver is installed and in your PATH."

    # try block tells the driver - if successfully initialized - to go to the careers URL we found
    # get BeautifulSoup to look it over for potential job titles
    try:
        driver.get(careers_url)

        # JavaScript-rendered job listings load AFTER the page finishes loading
        # introducing a wait time for everything to load before BeautifulSoup does its parsing
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a, h2, h3"))
        )

        # now scrape with BeautifulSoup
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        job_title_sanitized = sanitize_title(job_title)

        # create an empty list of potential job_titles against which we'll be checking our desired job
        potential_titles = []

        # go through the common HTML tags
        for tag in soup.find_all(['h2', 'h3', 'a', 'p', 'div']):
            text = tag.get_text(separator=' ', strip=True)
            # if the number of the words in the tag is more than 20, it's probably not a job title tag
            # the number '20' was suggested by Google
            if len(text.split()) <= 20:
                # add that job title to our list
                potential_titles.append(sanitize_title(text))

        # empty-results handling
        if not potential_titles:
            return f"Sorry. No job titles were found to match against. It may be due very latent rendering.\nCareers page: {careers_url}"

        # does the 'sanitized' job we're looking for match anything on the 'potential_titles' list? 
        # uses the fuzzywuzzy imports from above to identify similarities and gives us back a tuple with 
        # 2 pieces of information: 'best_match' and 'best_score'
        best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

        # 'best_score' from the 'fuzz.token_sort_ratio'
        if best_score >= 85:
            return f"✅ Looks legit! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
        else:
            return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

    except Exception as e:
        return f"❌ Seems like an error happened while we were scraping: {e}"

    # finally, close the browser
    finally:
        # if the driver was, indeed, successfully initialized:
        if driver: 
            driver.quit()

### <span style="color:blue">Step 2f: the 'run_validation' function</span>
> ##### Simply takes the user's input and guides the resulting functions

In [7]:
def run_validation():
    job_title_input = input("Enter the job title you want to check from the posting: ")
    company_name_input = input("Enter the name of the company: ")
    validation_result = validate_job_posting(job_title_input, company_name_input)
    print("\nValidation Result:")
    print(validation_result)

### <span style="color:blue">Step 3: Keep This Modular</span>
> ##### make sure this code only runs when needed

In [12]:
if __name__ == "__main__":
    run_validation()

Enter the job title you want to check from the posting:  Technical Sales and Field Service Engineer
Enter the name of the company:  CaptiveAire



Validation Result:
✅ Looks legit! 'Technical Sales and Field Service Engineer' ≈ 'technical sales field service engineer' (95% match)
Careers page: https://captiveaire.com/careers/


In [9]:
# import requests
# from bs4 import BeautifulSoup
# import re
# import time
# from urllib.parse import urlparse
# from typing import Optional
# from fuzzywuzzy import fuzz, process
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options

# def is_valid_url(url: str) -> bool:
#     try:
#         result = urlparse(url)
#         return all([result.scheme, result.netloc])
#     except ValueError:
#         return False

# def find_careers_page(company_name: str) -> Optional[str]:
    
#     # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
#     slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())

#     # not everything's a '.com'
#     possible_base_urls = [f"https://www.{slugged_company}.com", 
#                           f"https://www.{slugged_company}.net", 
#                           f"https://www.{slugged_company}.org"]
    
#     # typical career posting paths & subdomains
#     possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
#     possible_subdomains = ["careers.", "jobs."]

#     # make the server believe we're one of the more popular clients / browsers
#     # -> also, helps avoid blocks / captchas
#         headers = {
#         'User-Agent': (
#             'Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
#             "AppleWebKit/537.36 (KHTML, like Gecko) "
#             "Chrome/91.0.4472.124 Safari/537.36"
#             )
#         }
    
#     # loop for legitimacy and domain building
    
#     for base_url_attempt in possible_base_urls:
#         # stop the loop if the 'possible_base_url' seems wonky and skip to the try block
#         if not is_valid_url(base_url_attempt):
#             continue
#         # our plan if the 'possible_base_url' doesn't seem right
#         try:
#             # Send request to the main company URL and give it 7 seconds to respond
#             response = requests.get(base_url_attempt, headers=headers, timeout=7)
            
#             # No response after 7 seconds, check to see if we got an error message (like a 404 or 500 error)
#             response.raise_for_status()

#             # did the url redirect / somehow change after our request
#             we_got_redirected = response.url.lower() != base_url_attempt.lower()
            
#             # does that redirect have the career keywords we want?
#             career_keyword_in_redirect = any (
#                 keyword in response.url.lower()
#                 for keyord in ["careers", "jobs", "opportunities"]
#             )

#             # if that redirect has any of our 'career' keywords, then it's likely the page we want and we can stop searching
#             if we_got_redirected and career_keyword_in_redirect:
#                 return response.url

#             # check the HTML to see if server is running JavaScript redirects in my browser 
#             # -> important b/c 'requests' can't do JavaScript
#             soup = BeautifulSoup(response.text, 'html.parser')

#             # get soup to find any <meta> refresh HTML tags (look like '<meta http-equiv="refresh">')
#             # -> we're looking for are 'http-equiv' that matches a case-insensitive pattern of 'refresh'
#             meta_refresh = soup.find('meta', attrs={'http-equiv': re.compile(r'refresh', re.I)})
            
#             # if soup finds a <meta> refresh tag with a 'content' attribute:
#             if meta_refresh and 'content' in meta_refresh.attrs:
#                 # take that content text and store it as 'content'
#                 content = meta_refresh['content']
#                 # find anything in 'content' that looks like 'url=whatever' and store it 
#                 match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
#                 # if we get a match, store it as 'redirect_url'
#                 if match:
#                     redirect_url = match.group(2)
                    
#                     # break the redirect down into its components; and 
#                     # change the relative URL path into to an absolute URL path
#                     # -> '.netloc' == 'network location" == domain
#                     if not urlparse(redirect_url).netloc:
#                         redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                    
#                     # If the redirected URL looks valid; AND
#                     # the redirect contains "careers" or "jobs", give us the redirect url
#                     if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]): [cite: 3]
#                         return redirect_url

#             # JavaScript tag check -> Soup search for all occurences of '<script>' tags in the HTML 
#             script_tags = soup.find_all('script')

#             # loop through the tags we found
#             for script in script_tags:
#                 # check for and read the string content between the '<script></script>' tags
#                 # using regex to look to match the'window.location.href' string that indicates redirects
#                 if script.string:
#                     js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    
#                     # if there is a match, go to it (the redirect URL)
#                     if js_match:
#                         redirect_url = js_match.group(1)
#                         # change relative URLs to absolute URLs - like the '.netloc' note above
#                         if not urlparse(redirect_url).netloc:
#                             redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                        
#                         # If the redirected URL is valid and contains "careers" or "jobs", return it
#                         if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]): [cite: 3]
#                             return redirect_url

#        # what to do about that bad url from the beginning (see 'try')
#         except requests.exceptions.RequestException:
            
#             # If the initial request to the base URL fails, try the very next base domain or path w/ a 1-second delay
#             time.sleep(1)
#             continue
#     # create a url with the main url and 'extend' the possible paths:
#     # -> turn 'https://www.bobsbeds.com' into 'https://www.bobsbeds.com/careers' 
#     potential_urls = [base_url + path for path in possible_paths]
    
#     # kind of like the above when adding paths to the base URL,
#     # but this time with subdomains

# def sanitize_title(title: str) -> str:
#     title = re.sub(r'[^a-zA-Z0-9\s\-\|]', '', title)
#     title = re.sub(r'\s+', ' ', title).strip()
#     return title.lower()

# def validate_job_posting(job_title: str, company_name: str) -> str:
#     careers_url = find_careers_page(company_name)
#     if not careers_url:
#         return "Could not automatically find the company's careers page. Please check manually."

#     options = Options()
#     options.headless = True
#     driver = webdriver.Chrome(options=options)

#     try:
#         driver.get(careers_url)
#         soup = BeautifulSoup(driver.page_source, 'html.parser')

#         job_title_sanitized = sanitize_title(job_title)
#         potential_titles = []

#         for tag in soup.find_all(['h2', 'h3', 'a', 'p', 'div']):
#             text = tag.get_text(separator=' ', strip=True)
#             if len(text.split()) <= 20:
#                 potential_titles.append(sanitize_title(text))

#         best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

#         if best_score >= 85:
#             return f"✅ Fuzzy match found! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
#         else:
#             return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

#     except Exception as e:
#         return f"❌ An error occurred during scraping: {e}"

#     finally:
#         driver.quit()

# def run_validation():
#     job_title_input = input("Enter the job posting title from LinkedIn: ")
#     company_name_input = input("Enter the name of the company: ")
#     validation_result = validate_job_posting(job_title_input, company_name_input)
#     print("\nValidation Result:")
#     print(validation_result)

# if __name__ == "__main__":
#     run_validation()


In [14]:
# def find_careers_page(company_name: str) -> Optional[str]:
#     company_name_lower = company_name.lower().replace(' ', '')
#     base_url = f"https://www.{company_name_lower}.com"
#     possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
#     possible_subdomains = ["careers.", "jobs."]

#     potential_urls = [base_url + path for path in possible_paths]
#     potential_urls.extend([
#         base_url.replace("www.", subdomain)
#         for subdomain in possible_subdomains
#         if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
#     ])
#     potential_urls.append(f"https://{company_name_lower}.com/careers")

#     headers = {
#         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64) AppleWebKit/537.36 Chrome/91.0.4472.124 Safari/537.36'
#     }

#     for url in potential_urls:
#         if is_valid_url(url):
#             try:
#                 response = requests.get(url, headers=headers, timeout=5)
#                 response.raise_for_status()
#                 if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
#                     return response.url
#             except requests.exceptions.RequestException:
#                 time.sleep(1)

#     return None

### 

### 

In [13]:
#     # -> building the 'potential_urls' list with '.extend' instead of '.append' so we can add ALL the URLs (not just one)
#     potential_urls.extend([
#         # replace 'www' with the subdomain - end up with something like 'https://careers.google.com' instead of 'https://www.google.com'
#         base_url.replace("www.", subdomain)
#         for subdomain in possible_subdomains
#         # 
#         if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
#     ])
#     potential_urls.append(f"https://{company_name_lower}.com/careers")

#     # for loop to check possible career pages
#     for url in potential_urls:
#         # "if" the address is properly formatted
#         if is_valid_url(url):
#             try:
#                 response = requests.get(url, headers=headers, timeout=5)
#                 response.raise_for_status()
#                 if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
#                     return response.url
#             # if it is not...
#             except requests.exceptions.RequestException:
#                 time.sleep(1)

#     return None

